# Model Comparison — FDV RAG

Compare two Ollama models on a stratified sample from the test set.  
Retrieval runs **once per question**; both models receive identical context.

Set `MODEL_A`, `MODEL_B`, and `QUESTIONS_PER_LANG` in the config cell to run.

In [1]:
from pathlib import Path

MODEL_A = "qwen2.5:3b"
MODEL_B = "aya"

QUESTIONS_FILE     = Path("questions.csv")
QUESTIONS_PER_LANG = 4       # 4 × 3 languages = 12 total
SAMPLE_SEED        = 42

DATA_DIRS = {
    "de": Path("data/de"),
    "fr": Path("data/fr"),
    "it": Path("data/it"),
}
COLLECTION_NAME = "fdv_multilingual"
CHROMA_DIR      = Path("chroma_db_multilingual")

CFG = dict(TOP_K=3, BM25_WEIGHT=0.4, SEMANTIC_POOL=30, RERANK_POOL=6)

EMBEDDER_NAME      = "intfloat/multilingual-e5-small"
CROSS_ENCODER_NAME = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"

OLLAMA_TIMEOUT = 120   # seconds; raises an exception instead of hanging


In [2]:
import sys, time
from pathlib import Path
import pandas as pd
from IPython.display import display, HTML
import ollama
import chromadb
from sentence_transformers import SentenceTransformer, CrossEncoder
from tqdm.auto import tqdm

sys.path.insert(0, str(Path(".").resolve()))
from chunking import load_documents, build_chunk_records
from retrieval import build_bm25_index, retrieve
from generation import build_user_message, detect_language, SYSTEM_PROMPT


c:\Users\niw\Documents\CAS_NLP_Project\Project_tRAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Verify both models are pulled before spending time on retrieval
pulled = {m.model for m in ollama.list().models}

def _is_pulled(name):
    name_lc = name.lower()
    return any(
        p.lower() == name_lc or p.lower().startswith(name_lc + ":")
        for p in pulled
    )

missing = [m for m in (MODEL_A, MODEL_B) if not _is_pulled(m)]
if missing:
    cmds = "  ".join(f"ollama pull {m}" for m in missing)
    raise RuntimeError(
        f"Model(s) not found locally: {missing}\n"
        f"Pull them first:\n  {cmds}"
    )
print(f"Both models available: {MODEL_A}, {MODEL_B}")


Both models available: qwen2.5:3b, aya


In [4]:
print("Loading embedder...")
embedder = SentenceTransformer(EMBEDDER_NAME)
embedder.max_seq_length = 512

print("Loading cross-encoder...")
cross_encoder = CrossEncoder(CROSS_ENCODER_NAME)
print("Models ready.")


Loading embedder...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2125.34it/s]
BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading cross-encoder...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1593.86it/s]
XLMRobertaForSequenceClassification LOAD REPORT from: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Models ready.


In [5]:
chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection    = chroma_client.get_collection(COLLECTION_NAME)
print(f"Collection '{COLLECTION_NAME}': {collection.count()} chunks")

all_docs = []
for lang, data_dir in DATA_DIRS.items():
    all_docs.extend(load_documents(data_dir, lang))
chunk_records = build_chunk_records(all_docs)

bm25_index = build_bm25_index(chunk_records)
print(f"BM25 index ready ({len(chunk_records)} chunks).")


Collection 'fdv_multilingual': 4272 chunks
BM25 index ready (4272 chunks).


In [6]:
df_q = pd.read_csv(QUESTIONS_FILE)

df_sample = (
    df_q.groupby("Language")
        .sample(QUESTIONS_PER_LANG, random_state=SAMPLE_SEED)
        .reset_index(drop=True)
)

print(f"Sampled {len(df_sample)} questions ({QUESTIONS_PER_LANG} per language):")
display(df_sample[["ID", "Language", "Query", "Relevant section", "Target Term"]])

Sampled 12 questions (4 per language):


,ID,Language,Query,Relevant section,Target Term
0,1,de,Was macht ein Heizer?,"R 300.13, section 1.1",Feuerung
1,40,de,Wann muss der Lokführer die Wirkung der Luftbr...,"R 300.14, section 2.3.7",Gefälle
2,25,de,Wie können dem Lokführer die Zustimmung zur Fa...,"R 300.6, section 1.3.4",Kreuzung
3,4,de,Welche Arten von Rangierbewegungen gibt es?,"R 300.4, section 1.3",abstossen
4,65,fr,Quand est-il interdit d'enregistrer des itinér...,"R 300.4, section 2.3.7",lancer
5,41,fr,À quel moment le mécanicien doit-il s'assurer ...,"R 300.14, section 2.3.7",pente
6,26,fr,Comment l'assentiment pour circuler sur les tr...,"R 300.6, section 1.3.4",croisement
7,17,fr,Quels points peuvent être utilisés comme but d...,"R300.4, section 4.3.2",l'aiguille d'entrée
8,33,it,In quali condizioni è ammesso proseguire con u...,"R 300.9, section 12.3.4",carrozza
9,51,it,Dove si trova la soglia di velocità in stazion...,"R 300.6, section 2.2.2",primo scambio


## Generation

Retrieval runs once per question; both models receive identical context.

> **Before running:** make sure Ollama is serving (`ollama serve`) and both models
> are pulled (`ollama pull phi4-mini`, `ollama pull qwen2.5:3b`).

In [7]:
ollama_client = ollama.Client(timeout=OLLAMA_TIMEOUT)

results = []

for _, row in tqdm(df_sample.iterrows(), total=len(df_sample), desc="Generating"):
    query = row["Query"]

    # Retrieve once — identical context for both models
    chunks    = retrieve(query, embedder, cross_encoder, collection, bm25_index, chunk_records, CFG)
    lang_name = detect_language(query)
    user_msg  = build_user_message(query, chunks, lang_name)
    messages  = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_msg},
    ]

    answers = {}
    for model_name in (MODEL_A, MODEL_B):
        t0 = time.time()
        try:
            resp = ollama_client.chat(
                model=model_name,
                messages=messages,
                options={"temperature": 0.1},
            )
            text = resp.message.content
        except Exception as exc:
            text = f"[Timeout/Error after {OLLAMA_TIMEOUT}s: {exc}]"
        elapsed = time.time() - t0
        answers[model_name] = {"text": text, "seconds": round(elapsed, 1)}

    key_ans_a  = "answer_" + MODEL_A
    key_ans_b  = "answer_" + MODEL_B
    key_time_a = "time_"   + MODEL_A
    key_time_b = "time_"   + MODEL_B

    results.append({
        "id":               int(row["ID"]),
        "language":         row["Language"],
        "query":            query,
        "target_term":      str(row["Target Term"]),
        "relevant_section": str(row["Relevant section"]),
        "chunks":           chunks,
        key_ans_a:          answers[MODEL_A]["text"],
        key_ans_b:          answers[MODEL_B]["text"],
        key_time_a:         answers[MODEL_A]["seconds"],
        key_time_b:         answers[MODEL_B]["seconds"],
    })

print(f"Done. {len(results)} questions processed.")

Generating: 100%|██████████| 12/12 [38:34<00:00, 192.84s/it]

Done. 12 questions processed.


In [8]:
results_dir = Path("results")
results_dir.mkdir(exist_ok=True)

safe_a = MODEL_A.replace(":", "-").replace("/", "-")
safe_b = MODEL_B.replace(":", "-").replace("/", "-")
out_path = results_dir / f"model_comparison_{safe_a}_vs_{safe_b}.csv"

df_out = pd.DataFrame([{k: v for k, v in r.items() if k != "chunks"} for r in results])
df_out.to_csv(out_path, index=False, encoding="utf-8-sig")
print(f"Saved to {out_path}")


Saved to results\model_comparison_qwen2.5-3b_vs_aya.csv


## Results

Side-by-side answers for each question. Target term and expected section are shown for reference. Retrieved chunks are collapsible.

In [9]:
LANG_LABEL = {"de": "DE", "fr": "FR", "it": "IT"}

key_ans_a  = "answer_" + MODEL_A
key_ans_b  = "answer_" + MODEL_B
key_time_a = "time_"   + MODEL_A
key_time_b = "time_"   + MODEL_B


def _esc(s):
    return (str(s)
            .replace("&", "&amp;")
            .replace("<", "&lt;")
            .replace(">", "&gt;")
            .replace("\n", "<br>"))


for r in results:
    lang = r["language"]

    chunk_rows = ""
    for i, c in enumerate(r["chunks"], 1):
        reg   = c.get("regulation_number", "")
        sec   = c.get("section_id", "")
        ctext = _esc(c.get("text", "")[:300])
        chunk_rows += (
            f'<tr>'
            f'<td style="padding:4px;vertical-align:top;white-space:nowrap">'
            f'<b>Chunk {i}</b><br><small>{reg} §{sec}</small></td>'
            f'<td style="padding:4px">{ctext}…</td>'
            f'</tr>'
        )

    ans_a      = _esc(r[key_ans_a])
    ans_b      = _esc(r[key_ans_b])
    time_a     = r[key_time_a]
    time_b     = r[key_time_b]
    lang_label = LANG_LABEL.get(lang, lang.upper())

    html = (
        '<div style="border:1px solid #ccc;padding:12px;margin:16px 0;'
        'border-radius:6px;font-family:sans-serif">'
        f'<h4 style="margin:0 0 6px">[{lang_label}] Q{r["id"]}: {_esc(r["query"])}</h4>'
        f'<p style="margin:2px 0 8px">'
        f'<b>Target term:</b> <code>{_esc(r["target_term"])}</code> &nbsp;|&nbsp; '
        f'<b>Section:</b> {_esc(r["relevant_section"])}</p>'
        '<details style="margin:8px 0">'
        '<summary style="cursor:pointer;color:#555">Retrieved chunks (click to expand)</summary>'
        f'<table style="width:100%;font-size:0.85em;margin-top:6px;border-collapse:collapse">'
        f'{chunk_rows}</table></details>'
        '<table style="width:100%;border-collapse:collapse;margin-top:8px"><tr>'
        f'<th style="width:50%;padding:6px;background:#f0f0f0;border:1px solid #ddd;text-align:left">'
        f'{MODEL_A} &nbsp;<small>({time_a}s)</small></th>'
        f'<th style="width:50%;padding:6px;background:#f0f0f0;border:1px solid #ddd;text-align:left">'
        f'{MODEL_B} &nbsp;<small>({time_b}s)</small></th>'
        '</tr><tr>'
        f'<td style="padding:8px;vertical-align:top;border:1px solid #ddd">{ans_a}</td>'
        f'<td style="padding:8px;vertical-align:top;border:1px solid #ddd">{ans_b}</td>'
        '</tr></table></div>'
    )
    display(HTML(html))


Chunk 1R 300.13 §1,"1 Allgemeines1.1 Personal Als Heizer wird diejenige Person bezeichnet, der die Feuerung sowie bestimmte technische Aufgaben an der Dampflokomotive übernimmt. Er muss dafür fahrdienstlich nicht geprüft sein.Werden die Funktionen des Führergehilfen und des Heizers von einer einzigen Person wahrge…"
Chunk 2R 300.13 §2.2,2.2 Verantwortlichkeit Der LF ist für die Arbeit des Heizers mitverantwortlich. 2.3 Zuständigkeiten Der LF hat gegenüber dem Heizer Weisungsbefugnis.…
Chunk 3R 300.1 §2.5,"t die Alarmmittel aus. Die Ansteuerung erfolgt automatisch durch die Ankündigungsanlage oder manuellWarnsystemtechnische und/oder organisatorische Einrichtung, die Personen (bei Arbeiten im Gleisbereich) vor der Gefahr sich nähernden Fahrten warntWechselbetriebdie Ausrüstung jedes Streckengleise…"
qwen2.5:3b (60.3s),aya (120.0s)
"Ein Heizer ist die Person, die die Feuerung und bestimmte technische Aufgaben an der Dampflokomotive übernimmt. Er muss jedoch fahrdienstlich nicht geprüft sein.",[Timeout/Error after 120s: timed out]


Chunk 1R 300.14 §2.3.7,2.3.7 Bremsprobe auf Wirkung bei Zügen Der LF hat die Wirkung der Luftbremse zu prüfen– unmittelbar nach der Abfahrt– vom Ausgangs- oder Wendebahnhof– nach Veränderungen an der Zusammensetzung des Zuges– nach einem LF-Wechsel– vor der Einfahrt – in ein starkes Gefälle– in einen Kopfbahnhof…
Chunk 2R 300.14 §2.3.5,"remse auf 20 cmHg zu bringen. Anschliessend ist zu prüfen, ob die Bremszylinder Druckluft enthalten.– Vom Boden aus oder am Manometer für geschleppte oder in Vielfachsteuerung verkehrende Triebfahrzeuge in den Fällen, wo eine Hauptbremsprobe vorgeschrieben ist.Nach jedem Führerstandwechsel sind di…"
Chunk 3R 300.14 §2.2.5,"2.2.5 Winterbetrieb Im Winterbetrieb sind bei der Bremsbedienung mindestens folgende Massnahmen erforderlich:– Bei Temperaturen unter 0°C hat der LF unmittelbar nach Beginn der Fahrt das leichtgängige Rollen der Fahrzeuge zu prüfen– Nach dem Anhalten auf Bahnhöfen sind die Bremsen, wenn mög-lic…"
qwen2.5:3b (94.7s),aya (120.0s)
"Der LF hat die Wirkung der Luftbremse zu prüfen, wenn es nach Veränderungen an der Zusammensetzung des Zuges, nach einem LF-Wechsel oder vor der Einfahrt zur Prüfung erfolgt (FDV § 2.3.7).",[Timeout/Error after 120s: timed out]


Chunk 1R 300.6 §1.3.4,1.3.4 Zustimmung zur Fahrt auf Strecken ohne Hauptsignale Die Zustimmung zur Fahrt wird dem LF wie folgt erteilt:– mittels Fahrplans oder– mittels Fahrordnung oder– mittels Befehls für Kreuzung und Überholung oder– mündlich oder fernmündlich quittungspflichtig durch den FDL.Die ISB haben in i…
Chunk 2R 300.9 §2.4,2.4 Zustimmung2.4.1 Zustimmung bei Zügen Die Zustimmung wird wie folgt erteilt:– mit Fahrtstellung des Hauptsignals oder– mit Fahrtstellung des Hauptsignals durch eine Notbedienung oder– mit dem Hauptsignal gestörte Bahnübergangsanlage oder– mit dem Hilfssignal oder– mit dem protokollpflich…
Chunk 3R 300.4 §5.5,"5.5 Fahrt innerhalb der gesperrten Gleise5.5.1 Fahrt ohne Zustimmung Sofern keine Weichen befahren werden, kann innerhalb gesperrter Gleise auf eine Zustimmung verzichtet werden. Dabei müssen Hauptsignale, Rangiersignale und die Rangiergrenze nicht beachtet werden.Bei mehreren Rangierbewegungen…"
qwen2.5:3b (95.4s),aya (120.0s)
"Die Zustimmung zur Fahrt auf Strecken ohne Hauptsignale wird dem LF wie folgt erteilt:- mittels Fahrplans oder- mittels Fahrordnung oder- mittels Befehls für Kreuzung und Überholung oder- mündlich oder fernmündlich quittungspflichtig durch den FDL.Die ISB regeln in ihren Betriebsvorschriften das Vorgehen zur Erteilung, die Gültigkeit und die Bedingungen für die Rücknahme dieser Zustimmung.",[Timeout/Error after 120s: timed out]


Chunk 1R 300.4 §1.3,1.3 Bewegungsarten Es wird zwischen folgenden Rangierbewegungen unterschieden:– Rangierfahrten direkt und indirekt geführt– ablaufen lassen– abstossen– Rangierbewegungen mit Rangierseil oder Spill– Rangierbewegungen mit Strassenfahrzeugen– Rangierbewegungen von Hand oder mit mechanischen Hi…
Chunk 2R 300.4 §7.1.2,7.1.2 Bewegungsarten Es wird zwischen folgenden Rangierbewegungen unterschieden:- Rangierfahrten direkt und indirekt geführt- Rangierbewegungen mit Strassenfahrzeugen - Rangierbewegungen von Hand oder mit mechanischen Hilfsmitteln. …
Chunk 3R 300.4 §7,7 Rangierbewegungen im Trambetrieb 7.1 Vorbereitung und Abschluss 7.1.1 Leitung Jede Rangierbewegung wird von einem RL geleitet.Wechselt die Leitung der Rangierbewegung haben sich die RL gegenseitig zu orientieren. …
qwen2.5:3b (61.2s),aya (120.0s)
"Es werden folgende Rangierbewegungen unterschieden:- direkt und indirekt geführt (FDV 1.3, FDV 7.1.2)- ablaufen lassen- abstossen- mit Rangierseil oder Spill- mit Strassenfahrzeugen- von Hand oder mit mechanischen Hilfsmitteln (FDV 1.3, FDV 7.1.2)",[Timeout/Error after 120s: timed out]


Chunk 1R 300.4 §2.3.7,2.3.7 Enregistrement des itinéraires L’enregistrement d’itinéraires est interdit dans les cas suivants :– pour les courses qui suivent après un lancer– lorsque l’itinéraire à enregistrer touche le parcours de véhicules particuliers. …
Chunk 2R 300.4 §2.3.2,"2.3.2 Mouvement de manœuvre en direction d’itinéraire de train Sont considérés comme mouvement de manœuvre en direction d’un itinéraire de train établi, tous les mouvements de manœuvre qui peuvent mettre en danger la circulation des trains par une prise en écharpe à la suite d’un freinage tardif.…"
Chunk 3R 300.4 §2.9.3,2.9.3 Destruction des verrouillages et des itinéraires Le CC ne peut détruire les verrouillages ou les itinéraires que lorsque le mouvement de manœuvre est arrêté ou lorsque l’élément verrouillé est franchi.…
qwen2.5:3b (58.0s),aya (120.0s)
"L’enregistrement d'itinéraires est interdit pour les courses qui suivent après un lancer et lorsque l’itinéraire à enregistrer touche le parcours de véhicules particuliers (FDV, Section 2.3.7).",[Timeout/Error after 120s: timed out]


...

Chunk 1R 300.4 §4.3.2,"4.3.2 Buts du parcours Fait office de but :– un signal principal– un signal principal de la direction opposée– un signal de barrage ou un signal nain– une voie de raccordement– la voie de but en gare– l’aiguille d’entrée pour les gares sans signaux d’entrée ou, lorsqu’il est posé, l’indicat…"
Chunk 2R 300.4 §4.5.2,"4.5.2 Transmettre l’assentiment L’assentiment doit être transmis au CMAN par un ordre à quittancer avec le texte suivant : « pour (numéro) R, parcours protégé de (point de départ) à (but) ».Le point de départ et le but doivent être désignés de manière explicite et complétés en cas de nécessité a…"
Chunk 3R 300.4 §2.3,"2.3 Etablissement du parcours2.3.1 Conditions d’exploitation Avant d’établir le parcours, le CC doit s’assurer qu’aucun itinéraire de train ou autres mouvements de manœuvre ne soient mis en danger. Les aiguilles de protection et les sabots de déraillement suffisent comme mesures de sécurité. …"
qwen2.5:3b (82.0s),aya (120.0s)
"Les points suivants peuvent être utilisés comme but du parcours : signal principal, signal principal de la direction opposée, signal de barrage ou signal nain, voie de raccordement, voie de but en gare, aiguille d'entrée ou indicateur de début de gare.",[Timeout/Error after 120s: timed out]


Chunk 1R 300.9 §12.3.4,"12.3.4 Proseguimento con un veicolo non frenato in coda al treno La seguente procedura vale per al massimo un veicolo non frenato in coda al treno, se il freno deve essere disinserito durante la corsa, dopo la partenza alla stazione d’origine e al massimo fino alla prossima modifica della composi…"
Chunk 2R 300.5 §5.1.7,5.1.7 Veicolo non frenato in coda al treno Il trasferimento di un veicolo non frenato in coda al treno è ammesso unicamente se questo è provvisto di dispositivi di trazione e di repulsione funzionanti. A questo riguardo devono essere adempiute le condizioni secondo le prescrizioni «Perturbazioni …
Chunk 3R 300.5 §3.5.6,"3.5.6 Veicolo non frenato in coda al treno Il trasferimento di un veicolo non frenato in coda al treno è ammesso unicamente se questo, in seguito a un danno, è provvisto su un solo lato di organi di trazione e di repulsione funzionanti. A questo riguardo devono essere adempiute le condizioni sec…"
qwen2.5:3b (63.5s),aya (120.0s)
"È ammesso proseguire con un veicolo non frenato in coda al treno se il rapporto di frenatura e il minimo sono sufficienti per la velocità più bassa e la pendenza determinante, includendo massimamente due veicoli circolanti direttamente davanti.",[Timeout/Error after 120s: timed out]


Chunk 1R 300.6 §2.2.2,"2.2.2 Soglia di velocità in stazione Se sul lato d’entrata la velocità in stazione è diversa dalla velocità di tratta, la soglia di velocità si trova: • al primo scambio oppure • circa a 200 m dopo il segnale d’entrata, se nel binario interessato non ci sono scambi.Se sul lato d’uscita…"
Chunk 2R 300.6 §2.2.2,"all’ultimo scambio oppure • al segnale d’uscita, se nel binario interessato non ci sono scambi. Se la velocità d’entrata è diversa da quella d’uscita, la soglia di velocità si trova: • a metà del marciapiede del binario percorso oppure • a metà del fabbricato viaggiatori o, se manca …"
Chunk 3R 300.6 §2.2.2,", la posizione delle soglie di velocità deve essere regolata dal GI nelle prescrizioni d’esercizio. Se in stazione non sono prescritte velocità, in caso di via libera fra il primo e l’ultimo scambio vale la velocità di tratta più elevata ammessa prima della stazione o dopo la stessa. …"
qwen2.5:3b (63.5s),aya (120.0s)
"Se sul lato d'entrata la velocità in stazione è diversa dalla velocità di tratta, la soglia di velocità si trova:• al primo scambio oppure • circa a 200 m dopo il segnale d'entrata, se nel binario interessato non ci sono scambi.",[Timeout/Error after 120s: timed out]


Chunk 1R 300.7 §3.2,"3.2 Dati necessari Talune funzioni utilizzano curve di frenatura. Queste sono calcolate sulla scorta dei dati della tratta (funzione attiva, distanza, velocità finale, pendenza ecc.) e dei dati del veicolo (categoria di treno, rapporto di frenatura, lunghezza del treno ecc.). I dati della tratt…"
Chunk 2R 300.5 §3.2,"3.2 Calcolo di frenatura Il rapporto di frenatura necessario per determinare la categoria di freno è calcolato nel modo seguente: Regola per l’arrotondamento:nel calcolo di frenatura le frazioni di ton (t) e le percentuali di frenatura (%) di 0,5 e oltre vanno arrotondate per eccesso, quelle d…"
Chunk 3R 300.5 §1,1 Prescrizioni di frenatura1.1 Pesi-freno computabili1.1.1 Principio Possono essere computati tutti i freni a vuoto come pure i freni ad aria compressa azionati con il vuoto. …
qwen2.5:3b (58.3s),aya (120.0s)
"Le curve di frenatura sono calcolate utilizzando dati della tratta e del veicolo, come la funzione attiva, distanza, velocità finale, pendenza, categoria di treno, rapporto di frenatura e lunghezza del treno. I dati della tratta vengono programmati in base alla funzione richiesta e possono essere fischi o variabili. Il MAC rileva i dati del veicolo tramite un'interfaccia utente.",[Timeout/Error after 120s: timed out]


Chunk 1R 300.2 §1.2,"1.2 Rappresentazione1.2.1 Colore dei segnali Di regola, per i segnali ottici si usano i seguenti colori: rosso fermata, pericolo arancione prudenza, avvertimento, rallentamento verde via libera giallo servizio elettrico violetto segnali per treni Huckepack bianco segnali bassi, segnali per…"
Chunk 2R 300.2 §5.1.5,on la posa di una tavola bianca con bordo nero e con un punto nero Segnale principale sistema N Il segnale principale del sistema N è contrassegnato con un cerchio bianco catarifrangente. Il segnale può mostrare:– fermata– avvertimento– avviso di velocità– esecuzione di velocità– via lib…
Chunk 3R 300.2 §9,9 Segnali non valevoli o valevoli temporaneamente9.1 Segnali non valevoli9.1.1 Contrassegno Un segnale non valevole è contrassegnato con una croce bianco-nera oppure è coperto …
qwen2.5:3b (58.7s),aya (120.0s)
"Il colore bianco si utilizza per i segnali ottici che non hanno significato specifico, come segnali bassi o indicatori.",[Timeout/Error after 120s: timed out]


In [13]:
key_time_a = "time_" + MODEL_A
key_time_b = "time_" + MODEL_B

timing_rows = [
    {
        "ID":       r["id"],
        "Language": r["language"].upper(),
        f"{MODEL_A} (s)": r[key_time_a],
        f"{MODEL_B} (s)": r[key_time_b],
    }
    for r in results
]
df_times = pd.DataFrame(timing_rows)
display(df_times)

total_a = sum(r[key_time_a] for r in results)
total_b = sum(r[key_time_b] for r in results)
print(f"\nTotal {MODEL_A}: {total_a:.0f}s  ({total_a / len(results):.1f}s / question)")
print(f"Total {MODEL_B}: {total_b:.0f}s  ({total_b / len(results):.1f}s / question)")


,ID,Language,phi4-mini (s),qwen2.5:3b (s)
0,1,DE,63.1,53.5
1,40,DE,93.0,101.1
2,25,DE,55.3,63.9
3,4,DE,45.4,42.5
4,65,FR,55.4,45.9
5,41,FR,75.0,58.3
6,26,FR,61.8,67.5
7,17,FR,64.4,52.8
8,33,IT,84.0,62.2
9,51,IT,66.9,57.7



Total phi4-mini: 790s  (65.8s / question)
Total qwen2.5:3b: 709s  (59.1s / question)
